In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import ecdf
from natsort import natsorted
from os.path import join as pjoin
from tqdm.notebook import tqdm
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/spatial_info'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#00802d', 'B': '#006c79', 'C': '#004da4', 'D': '#430073'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## size of linear position bins equivalent to 2cm-wide bins
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Heatmaps of stability vs spatial information.

In [ ]:
data_type = 'S'
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'session': [], 'day': [], 'unit_id': [], 'odd_even': [], 
             'first_second': [], 'spatial_info': [], 'place_bool': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                index = ctn.mouse_indices(mouse, index)
                sdata = xr.open_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## remove neurons who didn't meet firing across trials criteria
                odd_even = sdata['odd_even'].values 
                first_second = sdata['first_second'].values 
                spatial_info = sdata['skaggs_info'].values
                unit_ids = sdata['unit_id'].values
                place_cells = sdata['skaggs_place'].values

                for cell in np.arange(0, sdata.shape[0]):
                    cell_dict['mouse'].append(mouse)
                    cell_dict['group'].append(sdata.attrs['group'])
                    cell_dict['sex'].append(sdata.attrs['sex'])
                    cell_dict['session'].append(sdata.attrs['session_two'])
                    cell_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    cell_dict['unit_id'].append(unit_ids[cell])
                    cell_dict['odd_even'].append(odd_even[cell])
                    cell_dict['first_second'].append(first_second[cell])
                    cell_dict['spatial_info'].append(spatial_info[cell])
                    cell_dict['place_bool'].append(place_cells[cell])
stability_df = pd.DataFrame(cell_dict)
avg_mouse = stability_df.groupby(['group', 'mouse', 'day'], as_index=False).agg({'odd_even': 'mean', 'first_second': 'mean'})
avg_stab = avg_mouse.groupby(['group', 'day'], as_index=False).agg({'odd_even': ['mean', 'sem'], 'first_second': ['mean', 'sem']})

In [ ]:
## Heatmap of stability on y axis, spatial information on x axis, z value is percentage (number of cells with those values) on day 16
nbins = 25
yvar = 'odd_even'
sub_df = stability_df[stability_df['day'] == 16]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_stab = np.linspace(np.min(sub_df[yvar]), np.max(sub_df[yvar]), nbins)

fig = pf.custom_graph_template(x_title='Spatial Information', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Two-context', 'Multi-context'], width=1000, height=500)
for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = sub_df[sub_df['group'] == group]
    H, _, _, = np.histogram2d(x=gdata['spatial_info'], y=gdata[yvar], bins=[bins_si, bins_stab])
    Hnorm = H / np.sum(H) * 100 ## in percent
    Hnorm[Hnorm == 0] = np.nan
    fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_stab[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=1, col=idx + 1) ## have to transpose H, see notes on histogram2d
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
if yvar == 'odd_even':
    fig.update_yaxes(title='Odd vs Even Trial Stability', col=1)
elif yvar == 'first_second':
    fig.update_yaxes(title='First vs Second Half Stability', col=1)
fig.update_yaxes(range=[-0.16, 1])
fig.show()
fig.write_image(pjoin(fig_path, f'{yvar}_mc_tc_day16_heatmaps_bins{nbins}.png'), width=1000, height=500)

In [ ]:
## Plot histograms from day 16 with only place cells
nbins = 25
yvar = 'odd_even'
sub_df = stability_df[(stability_df['day'] == 16) & (stability_df['place_bool'])]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_stab = np.linspace(np.min(sub_df[yvar]), np.max(sub_df[yvar]), nbins)

fig = pf.custom_graph_template(x_title='Spatial Information', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Two-context', 'Multi-context'], width=1000, height=500)
for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = sub_df[sub_df['group'] == group]
    H, _, _, = np.histogram2d(x=gdata['spatial_info'], y=gdata[yvar], bins=[bins_si, bins_stab])
    Hnorm = H / np.sum(H) * 100 ## in percent
    Hnorm[Hnorm == 0] = np.nan
    fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_stab[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=1, col=idx + 1)
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
if yvar == 'odd_even':
    fig.update_yaxes(title='Odd vs Even Trial Stability', col=1)
elif yvar == 'first_second':
    fig.update_yaxes(title='First vs Second Half Stability', col=1)
fig.update_yaxes(range=[-0.16, 1])
fig.show()
fig.write_image(pjoin(fig_path, f'{yvar}_mc_tc_day16_heatmaps_bins{nbins}_pcs_only.png'), width=1000, height=500)

In [ ]:
## Plot histograms from day 16 with only non-place cells
nbins = 25
yvar = 'odd_even'
sub_df = stability_df[(stability_df['day'] == 16) & (~stability_df['place_bool'])]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_stab = np.linspace(np.min(sub_df[yvar]), np.max(sub_df[yvar]), nbins)

fig = pf.custom_graph_template(x_title='Spatial Information', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Two-context', 'Multi-context'], width=1000, height=500)
for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = sub_df[sub_df['group'] == group]
    H, _, _, = np.histogram2d(x=gdata['spatial_info'], y=gdata[yvar], bins=[bins_si, bins_stab])
    Hnorm = H / np.sum(H) * 100 ## in percent
    Hnorm[Hnorm == 0] = np.nan
    fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_stab[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=1, col=idx + 1)
fig.update_layout(coloraxis1=dict(colorscale='magma_r'))
if yvar == 'odd_even':
    fig.update_yaxes(title='Odd vs Even Trial Stability', col=1)
elif yvar == 'first_second':
    fig.update_yaxes(title='First vs Second Half Stability', col=1)
fig.update_yaxes(range=[-0.28, 1])
fig.show()

In [ ]:
## Plot heatmaps for every day for Two-context mice
nbins = 25
yvar = 'odd_even'
group = 'Two-context'

sub_df = stability_df[stability_df['place_bool']]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_stab = np.linspace(np.min(sub_df[yvar]), np.max(sub_df[yvar]), nbins)
if yvar == 'odd_even':
    ytitle = 'Odd vs Even Trial Stability'
elif yvar == 'first_second':
    ytitle = 'First vs Second Half Stability'
fig = pf.custom_graph_template(x_title='Spatial Information (bits/event)', y_title=ytitle, rows=4, columns=5, shared_x=True, shared_y=True,
                               titles=day_list, width=1200, height=1200, master_axes=True)

for idx, day in enumerate(natsorted(sub_df['day'].unique())):
    if day == 21:
        pass 
    else:
        day_data = sub_df[(sub_df['day'] == day) & (sub_df['group'] == group)]
        
        if idx < 5:
            row, col = 1, idx + 1
        elif (idx >= 5) & (idx < 10):
            row, col = 2, idx - 4
        elif (idx >= 10) & (idx < 15):
            row, col = 3, idx - 9
        else:
            row, col = 4, idx - 14
        
        H, _, _, = np.histogram2d(x=day_data['spatial_info'], y=day_data[yvar], bins=[bins_si, bins_stab])
        Hnorm = H / np.sum(H) * 100 ## in percent
        Hnorm[Hnorm == 0] = np.nan
        fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_stab[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=row, col=col)
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
fig.update_yaxes(range=[np.min(sub_df[yvar]), 1])
fig.show()
fig.write_image(pjoin(fig_path, f'{group}_si_stab_heatmap_all_days_{nbins}_pcs_only.png'), width=1200, height=1200)

In [ ]:
## Plot heatmaps for every day for Multi-context mice
nbins = 25
yvar = 'odd_even'
group = 'Multi-context'

sub_df = stability_df[stability_df['place_bool']]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_stab = np.linspace(np.min(sub_df[yvar]), np.max(sub_df[yvar]), nbins)
if yvar == 'odd_even':
    ytitle = 'Odd vs Even Trial Stability'
elif yvar == 'first_second':
    ytitle = 'First vs Second Half Stability'
fig = pf.custom_graph_template(x_title='Spatial Information (bits/event)', y_title=ytitle, rows=4, columns=5, shared_x=True, shared_y=True,
                               titles=day_list, width=1200, height=1200, master_axes=True)

for idx, day in enumerate(natsorted(sub_df['day'].unique())):
    if day == 21:
        pass 
    else:
        day_data = sub_df[(sub_df['day'] == day) & (sub_df['group'] == group)]
        
        if idx < 5:
            row, col = 1, idx + 1
        elif (idx >= 5) & (idx < 10):
            row, col = 2, idx - 4
        elif (idx >= 10) & (idx < 15):
            row, col = 3, idx - 9
        else:
            row, col = 4, idx - 14
        
        H, _, _, = np.histogram2d(x=day_data['spatial_info'], y=day_data[yvar], bins=[bins_si, bins_stab])
        Hnorm = H / np.sum(H) * 100 ## in percent
        Hnorm[Hnorm == 0] = np.nan
        fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_stab[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=row, col=col)
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
fig.update_yaxes(range=[np.min(sub_df[yvar]), 1])
fig.show()
fig.write_image(pjoin(fig_path, f'{group}_si_stab_heatmap_all_days_{nbins}_pcs_only.png'), width=1200, height=1200)

In [ ]:
## Plot cumulative histograms of spatial information
ctx_colors = ['green', 'orange', 'darkorchid']
cell_type = 'all_cells'

nbins = 25
if cell_type == 'place_cells':
    sub_df = stability_df[stability_df['place_bool']]
elif cell_type == 'all_cells':
    sub_df = stability_df.copy()
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)

fig = pf.custom_graph_template(x_title='Spatial Information (bits/event)', y_title='Probability')
for idx, day in enumerate([6, 11, 16]):
    for group in ['Multi-context']:
        gdata = sub_df[(sub_df['day'] == day) & (sub_df['group'] == group)]
        H, xbins = np.histogram(gdata['spatial_info'], bins=bins_si)
        dx = xbins[1] - xbins[0]
        cumulative_H = np.cumsum(H) * dx
        cumulative_H = cumulative_H / np.max(cumulative_H)
        fig.add_trace(go.Scattergl(x=xbins, y=cumulative_H, mode='lines', line=dict(shape='hvh'),
                                   name=f'Day {day}', line_color=ctx_colors[idx]))
fig.show()

### Create trial rasters for an example mouse for an example session.

In [ ]:
bin_size = 0.06
binarized = True 
only_running = True
correct_dir = True

population_act, _ = pc.trial_raster(sdata, bin_size=bin_size, binarized=binarized, correct_dir=correct_dir, only_running=only_running)

In [ ]:
sorted_si = np.argsort(sdata['skaggs_info'].values)
uids = sorted_si[160:167]

fig = pf.custom_graph_template(x_title='', y_title='Trial', rows=uids.shape[0], columns=1, shared_x=True, shared_y=True,
                               width=500, height=1200, titles=[''])

for row_idx, uid in enumerate(uids):
    trial_raster = population_act[:, :, uid]

    fig.add_trace(go.Heatmap(x=bins[:-1], y=np.arange(0, trial_raster.shape[0]), z=(trial_raster > 0).astype(int),
                             coloraxis='coloraxis1', showscale=False), row=row_idx + 1, col=1)
fig.update_layout(coloraxis1=dict(colorscale='gray_r'))
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(title='Position (rad)', row=uids.shape[0])
fig.show()

In [ ]:
sorted_si = np.argsort(sdata['skaggs_info'].values)
uids = sorted_si[-10:-5]

fig = pf.custom_graph_template(x_title='', y_title='', rows=uids.shape[0], columns=1, shared_x=True, shared_y=True,
                               width=500, height=1200, titles=[''])

for row_idx, uid in enumerate(uids):
    trial_raster = population_act[:, :, uid]

    fig.add_trace(go.Heatmap(x=bins[:-1], y=np.arange(0, trial_raster.shape[0]), z=(trial_raster > 0).astype(int),
                             coloraxis='coloraxis1', showscale=False), row=row_idx + 1, col=1)
fig.update_layout(coloraxis1=dict(colorscale='gray_r'))
fig.update_yaxes(autorange='reversed')
fig.show()